# EDA — NPS Preditivo (Fase 1)

Análise exploratória **orientada a negócio** sobre pedidos, logística e atendimento.

- **Alvo:** `nps_score` (0 a 10), coletado após a jornada.
- **Tríade NPS (0–10):** Detrator ≤6, Neutro 7–8, Promotor ≥9.

Os gráficos são gravados em `reports/` para uso nos slides.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

_cwd = Path.cwd().resolve()
PROJECT_ROOT = _cwd if (_cwd / "data" / "raw" / "desafio_nps_fase_1.csv").exists() else _cwd.parent
RAW_PATH = PROJECT_ROOT / "data" / "raw" / "desafio_nps_fase_1.csv"
REPORTS = PROJECT_ROOT / "reports"
REPORTS.mkdir(parents=True, exist_ok=True)

assert RAW_PATH.exists(), f"CSV não encontrado: {RAW_PATH}"

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 120

df = pd.read_csv(RAW_PATH)

def nps_categoria(x: float) -> str:
    if x >= 9:
        return "Promotor"
    if x >= 7:
        return "Neutro"
    return "Detrator"

df["nps_categoria"] = df["nps_score"].map(nps_categoria)
df.head()

## 1. Qualidade dos dados

In [ ]:
print("Shape:", df.shape)
print("Duplicados order_id:", df["order_id"].duplicated().sum())
print("Nulos por coluna:")
print(df.isna().sum().sort_values(ascending=False).head(10))
df.info()

## 2. Distribuição do NPS (nota 0–10)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(df["nps_score"], bins=21, kde=True, ax=ax, color="#2E86AB")
ax.set_title("Distribuição do NPS (nota)")
ax.set_xlabel("nps_score")
plt.tight_layout()
fig.savefig(REPORTS / "fig01_hist_nps.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
order = ["Detrator", "Neutro", "Promotor"]
counts = df["nps_categoria"].value_counts().reindex(order)
fig, ax = plt.subplots(figsize=(6, 4))
counts.plot(kind="bar", ax=ax, color=["#C73E1D", "#F4A259", "#2A9D8F"])
ax.set_title("Volume por categoria NPS")
ax.set_ylabel("Pedidos")
plt.xticks(rotation=0)
plt.tight_layout()
fig.savefig(REPORTS / "fig02_categorias_nps.png", dpi=150, bbox_inches="tight")
plt.show()
counts

## 3. Logística: atraso é o fator mais crítico

O atraso na entrega apresenta associação forte com notas mais baixas (ver correlação no final). Aqui comparamos **média de NPS** por faixa de dias de atraso.

In [ ]:
bins = [-1, 0, 1, 3, 365]
labels = ["Sem atraso", "1 dia", "2 a 3 dias", "4+ dias"]
df["faixa_atraso"] = pd.cut(df["delivery_delay_days"], bins=bins, labels=labels)
tbl = df.groupby("faixa_atraso", observed=True)["nps_score"].agg(["mean", "median", "count"])
display(tbl)

fig, ax = plt.subplots(figsize=(8, 4))
sns.boxplot(data=df, x="faixa_atraso", y="nps_score", order=labels, ax=ax, palette="RdYlGn")
ax.set_title("NPS por faixa de atraso na entrega")
plt.xticks(rotation=15)
plt.tight_layout()
fig.savefig(REPORTS / "fig03_nps_por_atraso.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Reclamações e atendimento

Mais reclamações e mais contatos com atendimento costumam indicar **jornada problemática** e NPS menor.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.lineplot(data=df, x="complaints_count", y="nps_score", err_style="band", marker="o", ax=axes[0], color="#6A4C93")
axes[0].set_title("NPS médio × número de reclamações")
sns.lineplot(data=df, x="customer_service_contacts", y="nps_score", err_style="band", marker="o", ax=axes[1], color="#1982C4")
axes[1].set_title("NPS médio × contatos com atendimento")
plt.tight_layout()
fig.savefig(REPORTS / "fig04_reclamacoes_atendimento.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. CSAT interno vs NPS declarado

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
sns.scatterplot(data=df, x="csat_internal_score", y="nps_score", alpha=0.35, ax=ax)
ax.set_title("CSAT interno vs NPS")
plt.tight_layout()
fig.savefig(REPORTS / "fig05_csat_vs_nps.png", dpi=150, bbox_inches="tight")
plt.show()

df[["nps_score", "csat_internal_score"]].corr()

## 6. Região e tempo de relacionamento

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
order_reg = sorted(df["customer_region"].unique())
sns.barplot(data=df, x="customer_region", y="nps_score", order=order_reg, ax=axes[0], errorbar="ci", palette="pastel")
axes[0].set_title("NPS médio por região")
axes[0].tick_params(axis="x", rotation=20)

df["tenure_bin"] = pd.qcut(df["customer_tenure_months"], q=4, duplicates="drop")
sns.barplot(data=df, x="tenure_bin", y="nps_score", ax=axes[1], errorbar="ci", palette="crest")
axes[1].set_title("NPS por quartil de tenure (meses)")
axes[1].tick_params(axis="x", rotation=35)
plt.tight_layout()
fig.savefig(REPORTS / "fig06_regiao_tenure.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. “Ponto de ruptura” na experiência

In [ ]:
# Combinações operacionais críticas (linguagem para gestão)
df["ruptura_atraso"] = df["delivery_delay_days"] >= 4
df["ruptura_reclama"] = df["complaints_count"] >= 3

g = df.groupby(["ruptura_atraso", "ruptura_reclama"])["nps_score"].agg(["mean", "count"])
display(g.round(2))

fig, ax = plt.subplots(figsize=(7, 4))
plot_df = (
    df.assign(
        perfil=np.where(
            df["ruptura_atraso"] & df["ruptura_reclama"],
            "Atraso 4+ e 3+ reclamações",
            np.where(
                df["ruptura_atraso"],
                "Só atraso 4+ dias",
                np.where(df["ruptura_reclama"], "Só 3+ reclamações", "Demais pedidos"),
            ),
        )
    )
)
order_p = ["Demais pedidos", "Só 3+ reclamações", "Só atraso 4+ dias", "Atraso 4+ e 3+ reclamações"]
sns.barplot(data=plot_df, x="perfil", y="nps_score", order=order_p, ax=ax, palette="rocket")
ax.set_title("NPS médio por perfil de risco operacional")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
fig.savefig(REPORTS / "fig07_ponto_ruptura.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Mapa de correlações (variáveis numéricas)

In [ ]:
num = df.select_dtypes(include=[np.number]).drop(columns=["customer_id", "order_id"], errors="ignore")
corr = num.corr()
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr, annot=False, cmap="vlag", center=0, ax=ax)
ax.set_title("Correlação entre variáveis numéricas")
plt.tight_layout()
fig.savefig(REPORTS / "fig08_heatmap_corr.png", dpi=150, bbox_inches="tight")
plt.show()

corr["nps_score"].drop("nps_score").sort_values(key=abs, ascending=False).head(12)

## 9. Síntese para storytelling (gestão)

- **Problema visível:** maioria dos pedidos cai como **detrator** na tríade 0–10; poucos promotores.
- **Atraso:** pedidos **sem atraso** têm NPS médio muito superior aos de **4+ dias** de atraso — é o principal “alavanca” operacional nesta base.
- **Reclamações / atendimento:** cada reclamação a mais e cada contato extra pressionam o NPS para baixo.
- **CSAT interno** acompanha o NPS (bom sinal de consistência dos indicadores internos).
- **Região:** diferenças entre regiões são **modestas** frente ao efeito de atraso e reclamações — priorizar **execução logística e qualidade** antes de campanhas regionais genéricas.

Exportamos a base enriquecida para `data/processed/` (também reproduzível via `python scripts/prepare_data.py`).

In [ ]:
import subprocess
import sys

# Regenera CSV processado (mesma lógica que `python scripts/prepare_data.py`)
subprocess.check_call([sys.executable, str(PROJECT_ROOT / "scripts" / "prepare_data.py")], cwd=str(PROJECT_ROOT))
print("Base processada atualizada em data/processed/.")